# 05 Semantic model as code

Creates the Power BI semantic model **Nordklang Sales** on top of the gold tables in Direct
Lake mode, then adds relationships, the date table, sorting, hidden keys and the DAX measures.

Because the whole model is defined here, it can be rebuilt the same way in another workspace
(for example test and production) instead of being clicked together by hand. It uses
[Semantic Link Labs](https://github.com/microsoft/semantic-link-labs).

Run it once, and again when the model changes. It is not part of the daily pipeline: Direct
Lake reads the Delta tables directly, so new data shows up without an import refresh.

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
model_name = "Nordklang Sales"
lakehouse = "lh_retail"

In [ ]:
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

TABLES = {
    "Sales": "gold.fact_sales",
    "Targets": "gold.fact_target",
    "Date": "gold.dim_date",
    "Store": "gold.dim_store",
    "City": "gold.dim_city",
    "Product": "gold.dim_product",
    "Data quality": "meta.dq_latest",
}

RELATIONSHIPS = [  # many side, one side
    ("Sales", "date_key", "Date", "date_key"),
    ("Sales", "store_key", "Store", "store_key"),
    ("Sales", "product_key", "Product", "product_key"),
    ("Store", "city_key", "City", "city_key"),
    ("Targets", "city_key", "City", "city_key"),
    ("Targets", "date_key", "Date", "date_key"),
]

HIDDEN = {
    "Sales": ["date_key", "store_key", "product_key", "sales_line_id", "net_amount_local", "currency"],
    "Targets": ["date_key", "city_key"],
    "Date": ["date_key"],
    "Store": ["store_key", "city_key"],
    "City": ["city_key"],
    "Product": ["product_key"],
}

# name, folder, format, DAX
MEASURES = [
    ("Net Revenue", "Revenue", "#,0", "SUM ( Sales[net_revenue_dkk] )"),
    ("Gross Profit", "Revenue", "#,0", "SUM ( Sales[gross_profit_dkk] )"),
    ("Gross Margin %", "Revenue", "0.0%", "DIVIDE ( [Gross Profit], [Net Revenue] )"),
    ("Units", "Revenue", "#,0", "SUM ( Sales[quantity] )"),
    ("Avg Selling Price", "Revenue", "#,0", "DIVIDE ( [Net Revenue], [Units] )"),
    ("Return Rate %", "Revenue", "0.0%",
     "DIVIDE ( -CALCULATE ( [Net Revenue], Sales[is_return] = TRUE () ), CALCULATE ( [Net Revenue], Sales[is_return] = FALSE () ) )"),

    ("Net Revenue LY", "Time", "#,0", "CALCULATE ( [Net Revenue], SAMEPERIODLASTYEAR ( 'Date'[date] ) )"),
    ("Revenue Growth %", "Time", "+0.0%;-0.0%;0.0%", "DIVIDE ( [Net Revenue] - [Net Revenue LY], [Net Revenue LY] )"),
    ("Net Revenue FYTD", "Time", "#,0", "TOTALYTD ( [Net Revenue], 'Date'[date], \"05-31\" )"),
    ("Net Revenue FYTD LY", "Time", "#,0", "CALCULATE ( [Net Revenue FYTD], SAMEPERIODLASTYEAR ( 'Date'[date] ) )"),
    ("FYTD Growth %", "Time", "+0.0%;-0.0%;0.0%", "DIVIDE ( [Net Revenue FYTD] - [Net Revenue FYTD LY], [Net Revenue FYTD LY] )"),
    ("LFL Revenue Growth %", "Time", "+0.0%;-0.0%;0.0%", """
VAR ComparableStores =
    FILTER ( VALUES ( Store[store_key] ), [Net Revenue] > 0 && [Net Revenue LY] > 0 )
VAR Cur = CALCULATE ( [Net Revenue], ComparableStores )
VAR Prev = CALCULATE ( [Net Revenue LY], ComparableStores )
RETURN DIVIDE ( Cur - Prev, Prev )"""),

    ("Target", "Target", "#,0", "SUM ( Targets[target_dkk] )"),
    ("Target To Date", "Target", "#,0", """
VAR LastSaleKey = CALCULATE ( MAX ( Sales[date_key] ), ALL ( Sales ) )
RETURN CALCULATE ( [Target], KEEPFILTERS ( 'Date'[date_key] <= LastSaleKey ) )"""),
    ("Revenue vs Target %", "Target", "+0.0%;-0.0%;0.0%", "DIVIDE ( [Net Revenue] - [Target To Date], [Target To Date] )"),

    ("Active Stores", "Stores", "#,0", "DISTINCTCOUNT ( Sales[store_key] )"),
    ("Revenue per Store", "Stores", "#,0", "DIVIDE ( [Net Revenue], [Active Stores] )"),
    ("Online Share %", "Stores", "0.0%", "DIVIDE ( CALCULATE ( [Net Revenue], Store[channel] = \"Online\" ), [Net Revenue] )"),

    ("Blocking Failures", "Data quality", "0",
     "CALCULATE ( COUNTROWS ( 'Data quality' ), 'Data quality'[severity] = \"blocking\", 'Data quality'[status] = \"fail\" ) + 0"),
    ("Warnings", "Data quality", "0",
     "CALCULATE ( COUNTROWS ( 'Data quality' ), 'Data quality'[severity] = \"warning\", 'Data quality'[status] = \"fail\" ) + 0"),
    ("Quarantined Lines", "Data quality", "#,0", "CALCULATE ( SUM ( 'Data quality'[failed_rows] ), 'Data quality'[rule_id] = \"DQ11\" )"),
    ("Last Data Check", "Data quality", "yyyy-mm-dd hh:nn", "MAX ( 'Data quality'[checked_at] )"),
]

In [ ]:
labs.directlake.generate_direct_lake_semantic_model(
    dataset=model_name, tables=TABLES, source=lakehouse, source_type="Lakehouse",
    overwrite=True, refresh=False,
)

In [ ]:
with connect_semantic_model(dataset=model_name, readonly=False) as tom:
    for many_t, many_c, one_t, one_c in RELATIONSHIPS:
        tom.add_relationship(from_table=many_t, from_column=many_c, to_table=one_t, to_column=one_c,
                             from_cardinality="Many", to_cardinality="One", cross_filtering_behavior="OneDirection")
    tom.mark_as_date_table("Date", "date")
    tom.set_sort_by_column("Date", "month_name", "month_no")
    for table, cols in HIDDEN.items():
        for c in cols:
            tom.update_column(table_name=table, column_name=c, hidden=True)
    for table, col in [("Sales", "quantity"), ("Sales", "net_revenue_dkk"), ("Sales", "cost_dkk"),
                       ("Sales", "gross_profit_dkk"), ("Targets", "target_dkk")]:
        tom.update_column(table_name=table, column_name=col, hidden=True)  # use the measures instead
    for name, folder, fmt, dax in MEASURES:
        tom.add_measure(table_name="Sales", measure_name=name, expression=dax.strip(),
                        format_string=fmt, display_folder=folder)

labs.refresh_semantic_model(dataset=model_name)
print(f"{model_name}: {len(TABLES)} tables, {len(RELATIONSHIPS)} relationships, {len(MEASURES)} measures")

## Check the measures with DAX

In [ ]:
import sempy.fabric as fabric

check = fabric.evaluate_dax(model_name, """
EVALUATE
SUMMARIZECOLUMNS (
    'Date'[fiscal_year],
    "Net Revenue", [Net Revenue],
    "Gross Margin %", [Gross Margin %],
    "Revenue Growth %", [Revenue Growth %],
    "LFL Revenue Growth %", [LFL Revenue Growth %],
    "Revenue vs Target %", [Revenue vs Target %]
)
ORDER BY 'Date'[fiscal_year]
""")
display(check)